# RuO₂/TiO₂ epitaxy: strain coupling and vertical offset

Compare the `(0, 0, L)` CTR and adjacent $zDensity_G(z,0,0)$ profile for the `EpitaxyInterface` strain-coupling and film–bulk offset parameters.

- The RuO₂ film is approximately **15 nm** thick.
- The Skellam interface width is **3 nm**, converted to lower-bulk unit cells.
- The first sweep uses `κ = 0, 0.25, 0.5, 0.75, 1` at zero offset.
- The second sweep fixes `κ = 0` and varies the offset from `-1` to `+1` lower-bulk `c` lattice units.
- CTR curves are vertically separated by multiplicative factors; density curves are separated by additive offsets, following the other CTR comparison notebooks.


In [ ]:
%matplotlib widget
from pathlib import Path
import copy
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Find the checkout containing this example notebook."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "orgui" / "datautils").is_dir():
            return candidate
    return None


repository_root = find_repository_root()
if repository_root is not None:
    sys.path.insert(0, str(repository_root))

from orgui.datautils.xrayutils import CTRcalc

if (
    repository_root is not None
    and repository_root not in Path(CTRcalc.__file__).resolve().parents
):
    raise RuntimeError(
        "An installed orgui version is already loaded. Restart the "
        "kernel and run the notebook from the first cell."
    )


def find_example_file(filename):
    """Find an example file from the repository or notebook directory."""
    candidates = (Path(filename), Path("examples/CTR") / filename)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(filename)


model_path = find_example_file("RuO2_TiO2_Poisson_etching.xtal")
model_template = CTRcalc.SXRDCrystal.fromFile(model_path)
model_template.getUcNames()


## Model construction

`Film.basis[0]` is a number of structural layers, whereas `EpitaxyInterface.basis[0]` is a width in unit cells. The requested physical dimensions are converted using the RuO₂ structural-layer spacing and the TiO₂ lower-bulk `c` lattice constant, respectively.

The interface width is the Skellam standard deviation. A `tail_probability` of `1e-4` truncates only the far statistical tails while keeping the generated interface support shorter than the 15 nm film. The film surface is otherwise ideal; the Poisson surface component from the template is deliberately omitted so this notebook isolates epitaxial strain coupling and offset.


In [ ]:
film_thickness_A = 150.0
interface_width_A = 30.0
tail_probability = 1e-4

template_interface = model_template["TiO2toRuO2"]
template_film = model_template["RuO2"]

film_layer_spacing_A = (
    template_film.unitcell.a[2] / len(template_film.layers)
)
film_layers = int(round(film_thickness_A / film_layer_spacing_A))
actual_film_thickness_A = film_layers * film_layer_spacing_A
interface_width_cells = (
    interface_width_A / template_interface.uc_bottom.a[2]
)

print(f"Film: {film_layers} layers = {actual_film_thickness_A / 10:.3f} nm")
print(
    f"Interface width: {interface_width_cells:.4f} lower-bulk cells "
    f"= {interface_width_A / 10:.1f} nm"
)


def make_model(strain_coupling, offset=0.0):
    """Build a bulk–interface–film model for one parameter pair."""
    bulk = copy.deepcopy(model_template["bulk"])
    interface = copy.deepcopy(template_interface)
    film = copy.deepcopy(template_film)

    interface.profile = CTRcalc.SkellamProfile(
        width=interface_width_cells,
        asymmetry=0.0,
        tail_probability=tail_probability,
    )
    interface.basis[:] = [
        interface_width_cells,
        0.0,
        strain_coupling,
        offset,
    ]
    interface.basis_0[:] = interface.basis

    film.basis[0] = film_layers
    film.basis_0[:] = film.basis

    model = CTRcalc.SXRDCrystal(
        bulk,
        interface,
        film,
        stacking=np.array([1, 2]),
        atten=model_template.atten,
    )
    model.apply_stacking()
    return model


L = np.linspace(0.95, 7.0, 2400)
H = np.zeros_like(L)
K = np.zeros_like(L)
z = np.linspace(-150.0, 170.0, 6000)


def evaluate_sweep(values, model_factory):
    """Calculate `(0, 0, L)` amplitudes and real z densities."""
    ctrs = []
    densities = []
    for value in values:
        model = model_factory(value)
        ctrs.append(np.abs(model.F(H, K, L)))
        densities.append(np.real(model.zDensity_G(z, 0, 0)))
    return np.asarray(ctrs), np.asarray(densities)


def plot_paired_sweep(values, ctrs, densities, *, cmap_name, colorbar_label, title):
    """Plot vertically separated density and CTR curves in adjacent axes."""
    values = np.asarray(values, dtype=float)
    cmap = mpl.colormaps[cmap_name]
    norm = mpl.colors.Normalize(vmin=values.min(), vmax=values.max())
    ctr_stack_factor = 3.0
    density_step = 1.15 * np.max(np.abs(densities))

    fig, (density_ax, ctr_ax) = plt.subplots(
        1,
        2,
        figsize=(13.0, 5.4),
        constrained_layout=True,
    )
    for plot_index, (value, ctr, density) in enumerate(
        zip(values, ctrs, densities)
    ):
        color = cmap(norm(value))
        density_ax.plot(
            z,
            density + density_step * plot_index,
            color=color,
            linewidth=1.15,
        )
        ctr_ax.semilogy(
            L,
            ctr * ctr_stack_factor**plot_index,
            color=color,
            linewidth=1.2,
        )

    density_ax.axvline(0.0, color="0.25", linestyle="--", linewidth=0.9)
    density_ax.set_xlabel(r"$z$ / Angstrom")
    density_ax.set_ylabel(r"Vertically offset $\mathrm{Re}[\rho_{00}(z)]$")
    density_ax.set_title(r"Adjacent $zDensity_G(z,0,0)$ profiles")
    density_ax.grid(alpha=0.18)

    ctr_ax.set_xlabel(r"$L$ / r.l.u.")
    ctr_ax.set_ylabel(r"Vertically offset $|F_{00L}|$ / electrons")
    ctr_ax.set_title(r"$(0,0,L)$ CTR")
    ctr_ax.grid(alpha=0.18, which="both")
    fig.suptitle(title)

    colorbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=(density_ax, ctr_ax),
        pad=0.02,
        ticks=values,
    )
    colorbar.set_label(colorbar_label)
    return fig


## Strain coupling sweep

`strain_coupling = 0` leaves the bulk and film on their independent unstrained lattices. Increasing `strain_coupling` linearly moves their generated interface positions toward the fully strain-coupled field at `strain_coupling = 1`. The film–bulk offset remains zero.


In [ ]:
strain_couplings = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
strain_coupling_ctrs, strain_coupling_densities = evaluate_sweep(
    strain_couplings,
    lambda strain_coupling: make_model(strain_coupling=strain_coupling, offset=0.0),
)

strain_coupling_figure = plot_paired_sweep(
    strain_couplings,
    strain_coupling_ctrs,
    strain_coupling_densities,
    cmap_name="viridis",
    colorbar_label="Strain coupling κ",
    title="15 nm RuO₂ film, 3 nm interface: independent-lattice to fully strain-coupled",
)
# strain_coupling_figure


## Offset sweep at `strain_coupling = 0`

The offset is expressed in fractional lower-bulk `c` coordinates. It translates the RuO₂-side interface atoms and the complete film while leaving the TiO₂ bulk fixed. The sweep below spans `-1` to `+1`, including half-cell offsets.


In [ ]:
offsets = np.array([0.0, 0.1, -0.1, 0.2, -0.2])[::-1]
offset_ctrs, offset_densities = evaluate_sweep(
    offsets,
    lambda offset: make_model(strain_coupling=0.0, offset=offset),
)

offset_figure = plot_paired_sweep(
    offsets,
    offset_ctrs,
    offset_densities,
    cmap_name="plasma",
    colorbar_label=r"Offset / lower-bulk $c$",
    title="15 nm RuO₂ film, 3 nm interface: offset sweep at strain_coupling = 0",
)
# offset_figure


In [ ]:
offsets = np.array([0.0, 0.03, -0.03, 0.06, -0.06])[::-1]
offset_ctrs, offset_densities = evaluate_sweep(
    offsets,
        lambda offset: make_model(strain_coupling=1.0, offset=offset),
)

offset_figure = plot_paired_sweep(
    offsets,
    offset_ctrs,
    offset_densities,
    cmap_name="plasma",
    colorbar_label=r"Offset / lower-bulk $c$",
    title="15 nm RuO₂ film, 3 nm interface: offset sweep at strain_coupling = 1",
)
offset_figure